# Differential Gene Expression Analysis — UPS (26 Samples)

## Description

This notebook uses DESeq2 to compare gene expression between C1 (21 samples) and C2 (5 samples) in UPS.

Five samples were excluded from the original 31 (non-pleomorphic sarcomas, QC fail).

## Requirements

### Input Data

- `rsem.merged.gene_counts.tsv`: Raw RNA-seq gene count data.

### Required Software

- R with Jupyter Notebook (R kernel)
- DESeq2
- apeglm
- tidyverse

### Parameters

- Remove 5 samples from the original dataset.
- Keep genes with counts ≥ 30 in at least 5 samples.
- Compare C2 with C1 (C1 as the reference).
- Use `padj < 0.05` to identify significant genes.
- Use apeglm for log2 fold change shrinkage.

## Results

### 1. What important information do I gain from this analysis?

- Identify significantly differentially expressed genes between C1 and C2.
- Identify genes with higher expression in C1 or C2.

### 2. What plots, files etc. are created and where?

Three TSV files are saved in `../results/deseq2/`:

- `DESeq2_all_26samples.tsv`: Results for all tested genes.
- `C1_higher_26samples.tsv`: Significant genes with higher expression in C1.
- `C2_higher_26samples.tsv`: Significant genes with higher expression in C2.

No plots are generated in this notebook.

In [ ]:
# Differentialexpression


In [ ]:
# Load DESeq2
library(DESeq2)

In [ ]:
# Set input file paths
data_location = '~/Documents/ups/data/rnaseq/rsem/rsem.merged.gene_counts.tsv'

In [ ]:
# Load raw gene count data
counts_raw <- read.delim(data_location, check.names = FALSE)
# Preview the data
head(counts_raw)

In [ ]:
# Load packages for gene annotation and data manipulation
library(tidyverse)

In [ ]:
# Prepare count matrix for DESeq2
count_matrix <- counts_raw %>%
    select(-gene_name) %>%
    column_to_rownames("gene_id") %>%
    as.matrix()
# Convert counts to integers
count_matrix <- round(count_matrix)

In [ ]:
# Exclude 5 outlier samples
exclude_samples <- c(
    "PDXSAR010",
    "PDXSAR023",
    "SAR023",
    "SAR050",
    "SAR009"
)

count_matrix <- count_matrix[
    ,
    !(colnames(count_matrix) %in% exclude_samples)
]

In [ ]:
# Define Cluster 2 samples
cluster2_samples <- c(
    "PDXSAR008",
    "SAR008",
    "PDXSAR028",
    "SAR028",
    "SAR055"
)

In [ ]:
# Assign samples to C1 or C2
sample_info <- data.frame(
    sample = colnames(count_matrix),
    cluster = ifelse(
        colnames(count_matrix) %in% cluster2_samples,
        "C2",
        "C1"
    )
)
# Use sample names as row names
rownames(sample_info) <- sample_info$sample
# Check the number of samples in each cluster
table(sample_info$cluster)

In [ ]:
# Set C1 as the reference level
sample_info$cluster <- factor(
    sample_info$cluster,
    levels = c("C1", "C2")
)
# Create DESeq2 dataset
dds <- DESeqDataSetFromMatrix(
    countData = count_matrix,
    colData = sample_info,
    design = ~ cluster
)

In [ ]:
# Filter low-count genes
smallestGroupSize <- 5

keep <- rowSums(counts(dds) >= 30) >= smallestGroupSize

dds <- dds[keep, ]
# Check the dimensions after filtering
dim(dds)

In [ ]:
# Run DESeq2 analysis
dds <- DESeq(dds)

In [ ]:
# Check available DESeq2 result names
resultsNames(dds)

In [ ]:
# Extract differential expression results for C2 vs C1
res <- results(
    dds,
    name = "cluster_C2_vs_C1")
# Shrink log2 fold changes using apeglm
res <- lfcShrink(
    dds,
    coef = "cluster_C2_vs_C1",
    res = res,
    type = "apeglm"
)

In [ ]:
exists("dds")

In [ ]:
# Filter and sort significant DEGs
get_hits <- function(deseq_result){
    res_df <- as.data.frame(deseq_result)
# Keep significant genes
    filtered_sorted_res <- res_df[
        (!is.na(res_df$padj)) & (res_df$padj < 0.05),
    ]
# Count up- and down-regulated genes
    n_C2 <- sum(filtered_sorted_res$log2FoldChange > 0)
    n_C1 <- sum(filtered_sorted_res$log2FoldChange < 0)

    message(
        n_C1, " upregulated and ",
        n_C2, " downregulated hits (padj < 0.05)"
    )
   return(filtered_sorted_res)
}

In [ ]:
get_hits(res) %>% head(n=10)

In [ ]:
# Preview top 10 C1-higher significant genes
get_hits(res) %>% head(n = 10)

In [ ]:
# Add gene IDs and gene names to significant DEGs
hits <- get_hits(res)%>%
    rownames_to_column("gene_id") %>%
    left_join(counts_raw %>% select(gene_id, gene_name),by = "gene_id")
# Preview results
head(hits, 10)

In [ ]:
# Get C2-higher significant genes
hits_C2 <- hits %>%
filter(log2FoldChange > 0) %>%
arrange(desc(log2FoldChange))
# Preview top 10 C2-higher genes
head(hits_C2, 10)

In [ ]:
# Separate significant DEGs into C1-higher and C2-higher genes
hits_C1 <- hits %>%
    filter(log2FoldChange < 0) %>%
    arrange(log2FoldChange)

hits_C2 <- hits %>%
    filter(log2FoldChange > 0) %>%
    arrange(desc(log2FoldChange))

In [ ]:
# Check the dimensions of C1- and C2-higher gene sets
dim(hits_C1)
dim(hits_C2)

In [ ]:
# Save C1- and C2-higher genes for the 26-sample analysis
write.table(
    hits_C1,
    file = "../results/deseq2/C1_higher_26samples.tsv",
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
)

write.table(
    hits_C2,
    file = "../results/deseq2/C2_higher_26samples.tsv",
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
)

In [ ]:
# Export the COMPLETE DESeq2 result table for a volcano plot
# This is the same 26-sample DESeq2 run (C1 reference, C2 vs C1).
stopifnot(ncol(count_matrix) == 26)
stopifnot(sum(sample_info$cluster == "C2") == 5)

res_all <- as.data.frame(res) %>%
    tibble::rownames_to_column("gene_id") %>%
    dplyr::left_join(
        counts_raw %>% dplyr::select(gene_id, gene_name),
        by = "gene_id"
    )

output_file <- "../results/deseq2/DESeq2_all_26samples.tsv"
write.table(
    res_all,
    file = output_file,
    sep = "\t",
    quote = FALSE,
    row.names = FALSE
)
cat("Saved:", normalizePath(output_file), "\n")
cat("Number of genes:", nrow(res_all), "\n")